# ✈️ TRIP.com Price Crawler

Chạy lần lượt các cell từ trên xuống: **① Cấu hình → ② Đọc input → ③ Crawl → ④ Xem kết quả**.

> ⚠️ Trip chạy ở chế độ **BROWSER** (room API của Trip ký từng request nên không replay trực tiếp được — đã xác nhận). Chậm hơn Agoda nhưng ổn định.

**Đổi nguồn input:** sửa `INPUT_MODE` ở cell ① — `"gsheet"` (Google Sheet online) hoặc `"offline"` (file CSV/XLSX trên máy).

**Format file offline:** chỉ cần **3 cột đầu theo đúng thứ tự** `hotel_name, hotel_url, room_type` (tên cột không quan trọng, chỉ cần đúng thứ tự). Có file mẫu ở `input/TEMPLATE_hotels.csv`.

**Output** nằm trong `results/trip/`:
- `FINAL_<YYYYMMDD>.csv` — kết quả cuối
- `TEMP_trip.csv` — checkpoint: lỡ tắt giữa chừng, chạy lại cell ③ sẽ tự resume phần chưa xong

In [1]:
# ════════════════ ① CẤU HÌNH ════════════════

# ── Nguồn input: "gsheet" (online) hoặc "offline" (file trên máy) ──
INPUT_MODE = "gsheet"

# Dùng khi INPUT_MODE = "gsheet" (gid của tab được tự lấy từ URL)
GSHEET_URL = "https://docs.google.com/spreadsheets/d/1g_S06QeEAWnCTHYXGH0Nn4Mcb3FCT_-uIS1jUm4GGkw/edit?gid=607908359#gid=607908359"

# Dùng khi INPUT_MODE = "offline" — đường dẫn tuyệt đối, hoặc tương đối so với 31.crawl-tool
OFFLINE_FILE = "input/trip_hotels.csv"

# ── Tham số crawl ──
WEEKS      = 6      # số tuần cần crawl
MAX_HOTELS = 0      # 0 = crawl tất cả; đặt 5 để test nhanh 5 khách sạn đầu
SHARD      = ""     # "" = không chia; "1/2" = chạy phần 1 trong 2 phần (chạy lần lượt 1/2 rồi 2/2)

In [2]:
# ════════════════ ② ĐỌC INPUT ════════════════
import os, sys

if "ROOT" not in globals():                    # giữ nguyên ROOT khi chạy lại cell
    ROOT = os.path.abspath("")                 # .../31.crawl-tool (nơi đặt notebook này)
assert os.path.isdir(os.path.join(ROOT, "crawler")), (
    f"Không tìm thấy package `crawler` trong {ROOT} — hãy mở notebook từ thư mục 31.crawl-tool")
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import crawler
from crawler.hotels_io import read_hotels

if INPUT_MODE == "gsheet":
    INPUT = GSHEET_URL
    print("📡 Input: Google Sheet online")
else:
    INPUT = OFFLINE_FILE if os.path.isabs(OFFLINE_FILE) else os.path.join(ROOT, OFFLINE_FILE)
    assert os.path.exists(INPUT), f"Không tìm thấy file: {INPUT}"
    print(f"📁 Input: file offline — {INPUT}")

hotels = read_hotels(INPUT)
print(f"✅ Đọc được {len(hotels)} khách sạn. 5 dòng đầu:")
for name, url, room in hotels[:5]:
    print(f"   • {name} — {room}")

📡 Input: Google Sheet online
✅ Đọc được 55 khách sạn. 5 dòng đầu:
   • Halais Hotel — Superior Room
   • Minasi HanoiOi Hotel — Premier Double Bed Room
   • Muong Thanh Hanoi Centre Hotel — Superior King Room
   • Mercure Hanoi La Gare Hotel — Classic Double Bed Room With City View
   • REY Hotel Hanoi — Standard Twin Room


In [3]:
# (TÙY CHỌN) Tải Google Sheet về file offline — lần sau chỉ cần đổi INPUT_MODE = "offline"
import pandas as pd
from crawler.hotels_io import _gsheet_url

os.makedirs(os.path.join(ROOT, "input"), exist_ok=True)
dest = os.path.join(ROOT, "input", "trip_hotels.csv")
pd.read_csv(_gsheet_url(GSHEET_URL)).to_csv(dest, index=False, encoding="utf-8-sig")
print(f"💾 Đã lưu bản offline: {dest}")

💾 Đã lưu bản offline: /Users/hchinhtrung/Documents/GitHub/mvillage-email-template/31.crawl-tool/input/trip_hotels.csv


In [4]:
# ════════════════ ③ CRAWL ════════════════
OUTDIR = os.path.join(ROOT, "results", "trip")
os.makedirs(OUTDIR, exist_ok=True)
os.chdir(OUTDIR)                     # output (FINAL_*.csv, TEMP_trip.csv) nằm ở đây

kwargs = dict(
    site="trip",                     # browser-per-query (Trip không replay trực tiếp được)
    input=INPUT,
    weeks=WEEKS,
)
if MAX_HOTELS:
    kwargs["max"] = MAX_HOTELS
if SHARD:
    kwargs["shard"] = SHARD

await crawler.arun(**kwargs)         # notebook cho phép await trực tiếp

🩺 env problems (/Users/hchinhtrung/Documents/GitHub/mvillage-email-template/31.crawl-tool/.venv/bin/python):
  ⚠️ camoufox browser binary missing → python -m camoufox fetch
🚀 TRIP crawl | 55 hotels × 6w | browser-only | engine=camoufox | W1=2026-07-15

🏨 1/55 Halais Hotel | Superior Room | need weeks [1, 2, 3, 4, 5, 6]
      ✅ [Halais Hotel] W1: VND 1,082,584 (07/15, Superior Room)
      ✅ [Halais Hotel] W2: VND 1,082,584 (07/22, Superior Room)
      ✅ [Halais Hotel] W4: VND 1,082,584 (08/05, Superior Room)
      ✅ [Halais Hotel] W3: VND 1,082,584 (07/29, Superior Room)
      ✅ [Halais Hotel] W6: VND 1,235,295 (08/19, Superior Room)
      ✅ [Halais Hotel] W5: VND 1,082,584 (08/12, Superior Room)
   ✅ 6/6 priced (direct 0, fallback 6) | pace limit=3

🏨 2/55 Minasi HanoiOi Hotel | Premier Double Bed Room | need weeks [1, 2, 3, 4, 5, 6]
      ✅ [Minasi HanoiOi Hotel] W1: VND 1,467,393 (07/15, Premier Double Bed R)
      ✅ [Minasi HanoiOi Hotel] W2: VND 1,467,393 (07/22, Premier Double Bed

'FINAL_20260710.csv'

In [5]:
# ════════════════ ④ XEM KẾT QUẢ ════════════════
import glob
import pandas as pd

OUTDIR = os.path.join(ROOT, "results", "trip")
files = sorted(glob.glob(os.path.join(OUTDIR, "FINAL_*.csv")))
assert files, "Chưa có file FINAL nào — hãy chạy cell ③ trước."
latest = files[-1]
df = pd.read_csv(latest)
print(f"📄 {latest} — {len(df)} dòng")
df.head(20)

📄 /Users/hchinhtrung/Documents/GitHub/mvillage-email-template/31.crawl-tool/results/trip/FINAL_20260710.csv — 55 dòng


,hotel_name,room_type,price_w1,price_w2,price_w3,price_w4,price_w5,price_w6
0,Halais Hotel,Superior Room,"VND 1,082,584","VND 1,082,584","VND 1,082,584","VND 1,082,584","VND 1,082,584","VND 1,235,295"
1,Minasi HanoiOi Hotel,Premier Double Bed Room,"VND 1,467,393","VND 1,467,393","VND 1,467,393","VND 1,467,393","VND 1,467,393","VND 1,467,393"
2,Muong Thanh Hanoi Centre Hotel,Superior King Room,"VND 1,366,667","VND 1,366,667","VND 1,310,597","VND 1,310,597","VND 1,366,667","VND 1,310,597"
3,Mercure Hanoi La Gare Hotel,Classic Double Bed Room With City View,"VND 2,608,200","VND 2,381,400","VND 1,814,400","VND 1,723,680","VND 1,995,840","VND 1,723,680"
4,REY Hotel Hanoi,Standard Twin Room,"VND 2,500,000","VND 2,500,000","VND 2,300,000","VND 1,840,000","VND 2,300,000","VND 1,725,000"
5,La Passion Premium Cau Go,Grand Double Room,"VND 3,652,064","VND 3,652,064","VND 8,775,000","VND 8,775,000","VND 8,775,000","VND 8,775,000"
6,Bespoke Trendy Hotel Hanoi (Formerly Hanoi La ...,Cozy Deluxe Room,"VND 3,000,000","VND 1,521,669","VND 1,383,335","VND 1,844,447","VND 1,712,925","VND 1,844,447"
7,Nesta Hotel Hanoi,Superior Twin Room Non Smoking,"VND 1,200,719","VND 1,200,719","VND 1,200,719","VND 1,330,437","VND 1,297,176","VND 1,263,915"
8,Silk Path Boutique Hanoi,Superior Room,"VND 1,436,132","VND 1,436,132","VND 1,436,132","VND 1,520,610","VND 1,520,610","VND 1,520,610"
9,La Siesta Premium Hang Be,Superior Room No Window,"VND 2,439,360","VND 1,905,750","VND 1,905,750","VND 3,430,350","VND 2,668,050","VND 3,201,660"
